In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from scipy.stats import chi2, f
from panel_utils import ModelResultsAggregator, run_panel_regressions, run_spec_tests, run_panel_model_diagnostics
from test_fun import *
import pickle
from pathlib import Path
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
###############################
# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
###############################
df_reg_analys = pd.read_excel('cons_reg_analys.xlsx')
df_fed_analys = pd.read_excel('fed_analys.xlsx')

# Убираем служебные столбцы индекса
df_reg_analys = df_reg_analys.loc[:, ~df_reg_analys.columns.str.startswith('Unnamed')]
df_fed_analys = df_fed_analys.loc[:, ~df_fed_analys.columns.str.startswith('Unnamed')]

# Приводим даты к datetime
df_reg_analys['Date'] = pd.to_datetime(df_reg_analys['Date'])
df_fed_analys['Date'] = pd.to_datetime(df_fed_analys['Date'])
 
# Объединяем региональные и федеральные данные
fed_extra_cols = [
    col for col in df_fed_analys.columns
    if col not in df_reg_analys.columns and col != 'Region'
]
df_reg = df_reg_analys.merge(
    df_fed_analys[['Date'] + fed_extra_cols],
    on='Date',
    how='left'
)

df_reg['Cluster_3'] = ((df_reg['Cluster_1'] == 0) & (df_reg['Cluster_2'] == 0)).astype(int)


# Взаимодействия общего шока
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock'] * df_reg['Sank_dum']


# Взаимодействия негативного шока 
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_neg'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_neg'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_neg'] * df_reg['Sank_dum']

# Взаимодействия позитивного шока 
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_1' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl1'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_1']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl2'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_2']
if 'd_Mon_Shock' in df_reg.columns and 'Cluster_3' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Cl3'] = df_reg['d_Mon_Shock_pos'] * df_reg['Cluster_3']
if 'd_Mon_Shock' in df_reg.columns and 'Covid_dum' in df_reg.columns:
    df_reg['pos_d_Mon_Shock_Covid'] = df_reg['d_Mon_Shock_pos'] * df_reg['Covid_dum']
if 'd_Mon_Shock' in df_reg.columns and 'Sank_dum' in df_reg.columns:
    df_reg['neg_d_Mon_Shock_Sank'] = df_reg['d_Mon_Shock_pos'] * df_reg['Sank_dum']

# Кластерные выборки
if 'Cluster_1' in df_reg.columns and 'Cluster_2' in df_reg.columns:
    df_reg_clus_one = df_reg[df_reg['Cluster_1'] == 1].copy()
    df_reg_clus_two = df_reg[df_reg['Cluster_2'] == 1].copy()
    df_reg_clus_three = df_reg[df_reg['Cluster_3'] == 1].copy()


# Лаги 
df_reg["d_Int_Rate_ConsCred_lag1"] = df_reg["d_Int_Rate_ConsCred"].shift(1)

df_reg["d_Ex_Rate_lag1"] = df_reg["d_Ex_Rate"].shift(1)


################
# Убираем лишнее
################

cols_to_drop = ['Inflation_Expectations_trend', 'Inflation_Expectations_seasonal', 
        'Num_reg', 'exc_rate', 'exc_rate_diff','Ent_conf_ind_manufactoring', 'Key_Rate',
       'd_Key_Rate','Ent_conf_ind_manufactoring_adj', 'Nominal_Percent_Rate', 'Ent_conf_ind_mining',
       'Ent_conf_ind_manufactoring_trend',
       'Ent_conf_ind_manufactoring_seasonal', 'Ent_conf_ind_mining_adj',
       'Ent_conf_ind_mining_trend', 'Ent_conf_ind_mining_seasonal',
       'Cluster_3']
df_reg = df_reg.drop(columns = cols_to_drop, axis = 1, inplace = False)

# Модель

In [3]:
###########################
# Загрузить модель из файла
###########################

# models = load_models()
# model_name = "Модель 1"
# saved_spec = models.get(model_name)
# if saved_spec:
#     dependent_var = saved_spec["dependent_var"]
#     exog_vars_base = saved_spec["exog_vars_base"]
#     shock_vars = saved_spec["shock_vars"]
#     model_spec = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
#     exog_vars_small_1, exog_vars_small_2, exog_vars_small_3 = model_spec["exog_variants"]


### Создание модели

In [23]:
###############################
# Создание аргументов модели
###############################

dependent_var = 'd_Int_Rate_ConsCred'

exog_vars_base = [
    'd_Int_Rate_ConsCred_lag1',       # Лаг зависимой
    'New_Loans_ConsCred',           # Показатели портфеля
    'Zadolg_ConsCred',
    'Def_Zadolg_ConsCred', 
    'Cred_nagr',                    # Кредитная нагрузка
    'Fin_Dostup',                   # Доля фин орг
    'CAR_Indicator',                # CAR
    'D_top5_rozn',                  # Доля топ-5
    'Zakred',                       # Закредитованность
    'Exc_rate',                      # Валютный курс
    # 'd_Ex_Rate',                    
    'Bonds_Rate_Correct_5Y',        # Ставка по облигациям
    'Inflation_Expectations',       # Инфл ожидания
    # 'd_Inflation_Expectations',     
    'Covid_dum',                    # Дамми
    'Sank_dum',
    'Cluster_1',
    'Cluster_2',
]

shock_vars = [
    ['d_Mon_Shock_pos', 'd_Mon_Shock_neg'],
    ['ROISFIX'],
    ['MIACR'],
]

model_spec = build_shock_variants(dependent_var, exog_vars_base, shock_vars)
exog_vars_small_1, exog_vars_small_2, exog_vars_small_3 = model_spec["exog_variants"]


### Мультиколлинеарность

In [24]:
###############################
    #VIF-анализ - начальные панельные данные спецификация 1
###############################

# Tips to colinearity:
# 'Bonds_Rate_Correct_5Y' with 'CAR_Indicator'
# 'Bonds_Rate_Correct_5Y' and 'CAR_Indicator' with 'ROISFIX' and 'MIACR'
# 'Cred_nagr' with 'Zakred'
# 'Zadolg_ConsCred' with 'New_Loans_ConsCred'

vif_variables = [item for item in exog_vars_small_1 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred']]
df_vif = df_reg[vif_variables].copy()
df_vif = df_vif.dropna()

X_vif = sm.add_constant(df_vif[vif_variables])

# Расчет VIF
vif_data = pd.DataFrame()
vif_data["Variable"] = vif_variables
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i+1) for i in range(len(vif_variables))]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1")
print("="*60)
print(vif_data.to_string(index=False))


РЕЗУЛЬТАТЫ VIF АНАЛИЗА - НАЧАЛЬНЫЕ ПАНЕЛЬНЫЕ ДАННЫЕ СПЕЦИФИКАЦИЯ 1
                Variable      VIF
               Cluster_2 2.880375
                Sank_dum 2.827642
             D_top5_rozn 2.324796
                Exc_rate 2.286172
               Covid_dum 2.005442
               Cluster_1 1.995024
   Bonds_Rate_Correct_5Y 1.851992
              Fin_Dostup 1.755074
  Inflation_Expectations 1.710002
      New_Loans_ConsCred 1.558886
     Def_Zadolg_ConsCred 1.325945
                  Zakred 1.267962
         d_Mon_Shock_pos 1.148542
         d_Mon_Shock_neg 1.075356
d_Int_Rate_ConsCred_lag1 1.061688


In [25]:
exog_vars_small_1 = [item for item in exog_vars_small_1 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred']]
exog_vars_small_2 = [item for item in exog_vars_small_3 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred', 'Bonds_Rate_Correct_5Y']]
exog_vars_small_3 = [item for item in exog_vars_small_3 if item not in ['Cred_nagr','CAR_Indicator','Zadolg_ConsCred', 'Bonds_Rate_Correct_5Y']]

In [26]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

dependent_var = model_spec["dependent_var"]
exog_vars_initial = exog_vars_small_1

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_1 = pooled_res if pooled_success else None
fe_res_1 = fe_res if fe_success else None
re_res_1 = re_res if re_success else None


ПАНЕЛЬНАЯ РЕГРЕССИЯ: POOL + FE + RE (ОБЩАЯ ВЫБОРКА)
Зависимая переменная: d_Int_Rate_ConsCred

МОДЕЛЬ 1: POOLED OLS
                           PooledOLS Estimation Summary                          
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.1775
Estimator:                   PooledOLS   R-squared (Between):              0.3561
No. Observations:                 6159   R-squared (Within):               0.1764
Date:                 Tue, Jan 20 2026   R-squared (Overall):              0.1775
Time:                         14:51:50   Log-likelihood                -1.299e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      88.374
Entities:                           80   P-value                           0.0000
Avg Obs:                        76.987   Distribution:                 F(15,6144)
Min Obs:                        76.000                          

c:\PyProjects\CreditResearch\Monetary-policy-and-retail-landing-in-regions\panel_utils.py:1134: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

D_top5_rozn, Covid_dum, Sank_dum, Cluster_1, Cluster_2

  fe_res = fe_mod.fit(cov_type=cov_type, cluster_entity=cluster_entity)


                           PanelOLS Estimation Summary                           
Dep. Variable:     d_Int_Rate_ConsCred   R-squared:                        0.1814
Estimator:                    PanelOLS   R-squared (Between):             -181.57
No. Observations:                 6159   R-squared (Within):               0.1814
Date:                 Tue, Jan 20 2026   R-squared (Overall):             -0.8517
Time:                         14:51:51   Log-likelihood                -1.296e+04
Cov. Estimator:              Clustered                                           
                                         F-statistic:                      134.51
Entities:                           80   P-value                           0.0000
Avg Obs:                        76.987   Distribution:                 F(10,6069)
Min Obs:                        76.000                                           
Max Obs:                        77.000   F-statistic (robust):             168.76
                

In [ ]:
# run_panel_model_diagnostics(
#     y, X, pooled_res, fe_res, re_res,
#     pooled_success, fe_success, re_success
# )


In [ ]:
# # ===== ТЕСТЫ СПЕЦИФИКАЦИИ =====

# run_spec_tests(
#     y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success
# )


In [ ]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

dependent_var = model_spec["dependent_var"]
exog_vars_initial = exog_vars_small_2

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_2 = pooled_res if pooled_success else None
fe_res_2 = fe_res if fe_success else None
re_res_2 = re_res if re_success else None


In [ ]:
###############################
# Построение линейных моделей на панельных данных (КАК В ИССЛЕДОВАНИИ СКУРАТОВА & ЗВЕРЕВА)
# (Int_Rate_ConsCred зависимая переменная)
# Модель в первой разности - первый лаг ДКП
###############################

dependent_var = model_spec["dependent_var"]
exog_vars_initial = exog_vars_small_3

y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
    df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
)

# Сохраняем результаты модели для экспорта
pooled_res_3 = pooled_res if pooled_success else None
fe_res_3 = fe_res if fe_success else None
re_res_3 = re_res if re_success else None


In [ ]:
# # Optional: save model specs to Models.pkl
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 1")
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 2")
# save_model_spec(dependent_var, exog_vars_base, shock_vars, model_name="Модель 3")


### Вывод результатов

In [ ]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

dep_var_name = 'Int_Rate_ConsCred'
base_name = dep_var_name
if base_name.startswith(''):
    base_name = base_name[2:]
if base_name.endswith(''):
    base_name = base_name[:-4]

results_dir = ensure_results_dir('Results')

model_specs_all = [
    {
        'spec_name': 'Модель(м) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_1,
            'fe': fe_res_1,
            're': re_res_1
        }
    },
    {
        'spec_name': 'Модель(м) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_2,
            'fe': fe_res_2,
            're': re_res_2
        }
    },
    {
        'spec_name': 'Модель(м) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_3,
            'fe': fe_res_3,
            're': re_res_3
        }
    },
    {
        'spec_name': 'Модель(б) шок Тейлора',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_4,
            'fe': fe_res_4,
            're': re_res_4
        }
    },
    {
        'spec_name': 'Модель(б) ROISFIX',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_5,
            'fe': fe_res_5,
            're': re_res_5
        }
    },
    {
        'spec_name': 'Модель(б) MIACR',
        'dependent_var': dep_var_name,
        'subsample': 'Общая выборка',
        'results': {
            'pooled': pooled_res_6,
            'fe': fe_res_6,
            're': re_res_6
        }
    }
]

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

out_all = os.path.join(results_dir, f"{base_name}_test.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)

In [ ]:
models_dict = load_models()
models_table = pd.DataFrame.from_dict(models_dict, orient='index')
models_table


### Выгрузка всех моделей


In [ ]:
from model_results_export import ModelResultsAggregator, ensure_results_dir, add_model_set, build_and_export
import os

models_dict = load_models()

model_specs_all = []
for model_name, saved_spec in models_dict.items():
    model_spec = build_shock_variants(
        saved_spec["dependent_var"],
        saved_spec["exog_vars_base"],
        saved_spec["shock_vars"],
    )
    exog_variants = model_spec["exog_variants"]

    for idx, exog_vars_initial in enumerate(exog_variants):
        shock_entry = saved_spec["shock_vars"][idx]
        if isinstance(shock_entry, (list, tuple)):
            shock_name = "+".join(shock_entry)
        else:
            shock_name = str(shock_entry)

        dependent_var = model_spec["dependent_var"]
        y, X, pooled_res, fe_res, re_res, pooled_success, fe_success, re_success = run_panel_regressions(
            df_reg, df_reg, dependent_var, exog_vars_initial, cov_type='clustered', cluster_entity=True, df_name='df_reg'
        )

        model_specs_all.append({
            'spec_name': f"{model_name} - {shock_name}",
            'dependent_var': dependent_var,
            'subsample': 'Общая выборка',
            'results': {
                'pooled': pooled_res if pooled_success else None,
                'fe': fe_res if fe_success else None,
                're': re_res if re_success else None
            }
        })

aggregator_all = ModelResultsAggregator()
for spec in model_specs_all:
    add_model_set(aggregator_all, spec)

results_dir = ensure_results_dir('Results')
date_tag = pd.Timestamp.today().strftime('%d_%m_%y')
out_all = os.path.join(results_dir, f"all_models_{date_tag}.xlsx")
build_and_export(aggregator_all, out_all, include_pvalues=True, decimals=3)
